## Loading and unlocking the statement

In [1]:
from dotenv import load_dotenv
from pathlib import Path
import pikepdf
import os

load_dotenv()

BASE_DIR = Path.cwd().parent

pdf_input = BASE_DIR / "resources" / "mpesa_statement.pdf"
pdf_unlocked = BASE_DIR / "resources" / "unlocked_statement.pdf"

pdf_password = os.getenv("FILE_PASSWORD")

with pikepdf.open(pdf_input, password=pdf_password) as pdf:
    pdf.save(pdf_unlocked)

In [2]:
# Extracting tables from PDF
import pandas as pd
import pdfplumber

all_rows = []

with pdfplumber.open(pdf_unlocked) as pdf:
    for page in pdf.pages:
        table = page.extract_table()
        
        if table:
            for row in table:
                all_rows.append(row)

In [3]:
# Converting to dataframe
df = pd.DataFrame(all_rows)

In [4]:
# Setting column names
df.columns = [
    "Receipt No",
    "Completion Time",
    "Details",
    "Transaction Status",
    "Paid In",
    "Withdrawn",
    "Balance"
]

## Removing rows with header keywords (page headers)

In [5]:
header_values = ["Receipt No", "Completion Time", "Details", "Transaction Status", "Paid In", "Withdrawn", "Balance"]

matching_rows = df[df.apply(lambda row: all(any(h.lower() in str(cell).lower() for h in header_values) for cell in row), axis=1)]

len(matching_rows)

112

In [6]:
df = df[~df.apply(lambda row: all(any(h.lower() in str(cell).lower() for h in header_values) for cell in row), axis=1)].reset_index(drop=True)

## Cleaning details column (split into multiple lines)

In [7]:
# Merging broken lines
def clean_details(text):
    if pd.isna(text):
        return ""
    return " ".join(str(text).split())

df["Details"] = df["Details"].apply(clean_details)

## Extracting relevant details

In [8]:
import re

def extract_notes(text):
    if pd.isna(text):
        return None
    
    text = str(text)
    
    # getting everything after "-"
    match = re.search(r"-\s*(.+)", text)
    if not match:
        return None
    
    segment = match.group(1)
    # cutting off trailing noise
    segment = re.split(r"\.| for | Acc\.|Original|by|via", segment)[0]
    # removing numbers (masked or full)
    segment = re.sub(r"\b\d[\d\*]*\b", "", segment)
    # cleaning spaces
    segment = " ".join(segment.split())
    
    return segment.strip() if segment else None

df["Notes"] = df["Details"].apply(extract_notes)

## Processing the time column

In [9]:
df['Completion Time'].dtype

dtype('O')

In [10]:
# Converting to datetime
df['Completion Time'] = pd.to_datetime(df['Completion Time'], errors='coerce')

In [11]:
df['Completion Time'].dtype

dtype('<M8[ns]')

In [12]:
# time-based features
df['Year'] = df['Completion Time'].dt.year
df['Month'] = df['Completion Time'].dt.strftime('%b')
df['Date'] = df['Completion Time'].dt.day
df['Day'] = df['Completion Time'].dt.day_name()

## Cleaning numeric columns

In [13]:
def extract_amount(value):
    if pd.isna(value):
        return 0.0
    value = str(value)
    value = value.replace("-", "").strip()
    try:
        return float(value)
    except ValueError:
        return 0.0

df["Paid In"] = df["Paid In"].apply(extract_amount)
df["Withdrawn"] = df["Withdrawn"].apply(extract_amount)
df["Balance"] = df["Balance"].apply(extract_amount)

## Creating amount column

In [14]:
df["Amount"] = df["Paid In"] - df["Withdrawn"]

## Categorizing transactions as income or expense based on amount

In [15]:
df["Category"] = df["Amount"].apply(lambda x: "Income" if x > 0 else ("Expense" if x < 0 else "Neutral"))

## Saving clean CSV

In [16]:
cleaned_transactions = BASE_DIR / "resources" / "clean_mpesa_transactions.csv"

df.to_csv(cleaned_transactions, index=False)